# Feature Engineering Techniques

**Source:** LinkedIn Learning Course  
**Topic:** Feature Engineering - Transformation and Creation Strategies  
**Status:** Exploratory notebook for learning feature engineering patterns

## Overview

This notebook demonstrates core feature engineering techniques for improving ML model performance. Feature engineering is the process of creating new features or transforming existing ones to better capture patterns in data.

## Key Techniques Covered

### Data Transformation
- **Scaling/Normalization** - MinMaxScaler, StandardScaler
- **Log Transformation** - Handle skewed distributions
- **Power Transformations** - Box-Cox, Yeo-Johnson for normality

### Feature Creation
- **Polynomial Features** - Interaction terms (x₁ × x₂, x₁²)
- **Binning/Discretization** - Convert continuous to categorical
- **Date/Time Features** - Extract year, month, day, weekday
- **Domain-Specific Features** - MPG ratios, efficiency metrics

### Feature Selection
- **Correlation Analysis** - Remove highly correlated features
- **Variance Thresholds** - Remove low-variance features
- **Mutual Information** - Select features with high signal

## Dataset

Uses EPA Fuel Economy data with features:
- **Numeric**: `cylinders`, `displ` (displacement), `city08` (city MPG), `highway08`, `comb08` (combined MPG)
- **Categorical**: `make`, `model`, `VClass`, `fuelType`, `drive`
- **Target**: `barrels08` (annual petroleum consumption)

## Visualizations Included

- Distribution plots (before/after transformations)
- Correlation heatmaps
- Feature importance charts
- Scatter plots with polynomial fits

## Notes for Future Revision

**Security & Data Management:**
- ✅ Loads data from public EPA API (no credentials)
- ✅ No hardcoded secrets
- ✅ Timezone parsing handled correctly

**Key Learnings:**
- **Transform before split**: Apply transformations only on training data, then apply same transformation to test (prevent leakage)
- **Polynomial features explode**: Degree 2 with 10 features → 55 features. Use sparingly
- **Log transforms**: Only work for positive values. Add small constant for zero values
- **Domain knowledge matters**: MPG ratios, efficiency metrics often outperform raw features

**Production Patterns:**
- Use sklearn Pipeline to chain transformations (prevents leakage)
- Save fitted transformers (pickle/joblib) for inference
- Monitor feature distributions in production (detect drift)
- Version control feature engineering code (reproducibility)

**Related to AI Portfolio:**
- Connects to notes/01-ml regression and classification chapters
- Feature engineering often provides bigger gains than model tuning
- Preprocessing patterns apply to both ML and AI pipelines

**Improvements Needed:**
- [ ] Add sklearn Pipeline examples (proper train/test workflow)
- [ ] Show before/after model performance metrics
- [ ] Add feature importance analysis (which engineered features help most)
- [ ] Document computational cost of each transformation
- [ ] Add automated feature selection examples (RFE, SelectKBest)
- [ ] Include cross-validation in feature selection

**Dependencies:**
- pandas, numpy, matplotlib, seaborn
- scikit-learn (preprocessing, feature_selection)
- EPA vehicles API (public)

**Common Pitfalls to Avoid:**
- ❌ Fitting transformers on entire dataset (leaks test info into training)
- ❌ Creating too many polynomial features (curse of dimensionality)
- ❌ Ignoring missing values before transformation
- ❌ Not saving fitted transformers for production inference

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import numpy as np
import pandas as pd
from utils import load_fuel_economy_data

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

In [ ]:
df.isna().mean()*100

In [ ]:
# find all vehicles that have missing cylinders info
df[df['cylinders'].isna()]

# This is likely going to be EVs where the data is expected to be missing
# To "clean" our data we will want to set this value to something meaningful
# Usually we consult SMEs to understand what would be the neutral state of the data
# when values are missing
# In this case 0 is the right answer but it could easily be mean or median depending
# on the semantic nature of the data

In [ ]:
# assign the missing values to 0
df.assign(cylinders=df.cylinders.fillna(0))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn import set_config

# set output to pandas
set_config(transform_output='pandas')

# create a pipeline for cylinders
cyl_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0)) # assign the default value to 0 for missing values
])

cyl_pipe.fit_transform(df[['cylinders']])


In [ ]:
# see where it "filled" the missing values

(cyl_pipe
.fit_transform(df[['cylinders']]) # the transformed values
.loc[df.cylinders.isna()] # from the original dataframes all the na columns for cylinders
)

In [ ]:
# The prod like pipeline if we really decided to use cylinder and displacement columns

from sklearn.compose import ColumnTransformer

# create the imputers
cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0) # fill missing values with 0
displ_imputer = SimpleImputer(strategy='median') # fill missing values with median


# create the preprocessors
preprocessor = ColumnTransformer(
    transformers=[
        ('cyl_imputer', cylinders_imputer, ['cylinders']), # creates a new columnn with the transform applied
        ('displ_imputer', displ_imputer, ['displ'])
    ],
    remainder='passthrough' # passes through all the existing columns
)

# create the pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

# fit and transform the data
pipeline.fit_transform(df)


In [ ]:
import numpy as np
df.city08.hist()


In [ ]:

# log hist confirms the long tail
np.log(df.city08).hist()

In [ ]:
from scipy import stats
from matplotlib import pyplot as plt


stats.probplot(df.city08, plot=plt)
# the overall plot doesn't align with the normal distribution straight line

In [ ]:
stats.probplot(df[df['city08']<40].city08, plot=plt)
# the filtered subset does a much better job of aligning

In [ ]:
# There are a few ways we could handle this skewed data

# 1. ignore the skewed tail as noisy outliers but this is risky what if the tail is actually a strong signal
# like luxury/sports vehicles with low mileage, model should work against them too

# 2. Bin the data so we can address the normal distribution shaped data
# if we use equally spaced bins like below we won't get a normal distribution over bins

from sklearn.preprocessing import KBinsDiscretizer

column_transformer = ColumnTransformer(
    transformers = [
        ('binning', KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile'), ['city08'])
    ],
    remainder = 'passthrough'
)

pipeline = Pipeline(steps=[('transformer', column_transformer)])
pipeline.fit_transform(df).binning__city08.value_counts().sort_index().plot.bar()


In [ ]:
cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
displ_imputer = SimpleImputer(strategy='median')
binning_strategy = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

column_transformer = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']), # creates a new columnn with the transform applied
        ('displ_imputer', displ_imputer, ['displ']),
        ('binning', binning_strategy, ['city08'])
    ],
    remainder='passthrough'
)

pipeline = Pipeline(steps=[('transformer', column_transformer)])
pipeline.fit_transform(df)

# Transforms, binning or other strategies are a judgement call, tune as needed, a lot of the times
# linear regression works well without this tuning

In [ ]:
# check how the log of the values behave wrt the original values
(
df
.assign(city08_log=np.log(df.city08))
.plot.scatter(x='city08_log', y='city08')
)

In [ ]:

# also check the hist to see the distribution of values
df.city08.hist()

In [ ]:
np.log(df.city08).hist()

In [ ]:
# Build a linear regression model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X=df.drop(columns=['city08', 'highway08', 'comb08']).select_dtypes('number')
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

cylinders_imputer = SimpleImputer(strategy='constant', fill_value=0)
displ_imputer = SimpleImputer(strategy='median')

column_transformer = ColumnTransformer(
    transformers = [
        ('cyl_imputer', cylinders_imputer, ['cylinders']), # creates a new columnn with the transform applied
        ('displ_imputer', displ_imputer, ['displ'])
    ],
    remainder='passthrough'
)

# train the model
pipeline = Pipeline(steps=[('preprocessor', column_transformer), ('lr', LinearRegression())])
pipeline.fit(X_train, y_train)


# evaluate the model
pipeline.score(X_test, y_test) # r2 score



In [ ]:
from sklearn.metrics import mean_absolute_error
# MSE
mean_absolute_error(y_test, pipeline.predict(X_test))

In [ ]:
# use log transform to model now
y_log=np.log(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_log, random_state=42)

pipeline_log = Pipeline(steps=[('transformer', column_transformer), ('lr', LinearRegression())])
pipeline_log.fit(X_train, y_train)

print(f"R2 score is {pipeline_log.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline_log.predict(X_test))}")

In [ ]:
(
df
.assign(barrels08_log=np.log(df.barrels08))
.plot.scatter(x='barrels08_log', y='city08')
)

In [ ]:
df.barrels08.hist()


In [ ]:
df.barrels08.isna().sum()

In [ ]:
# Build a linear regression model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X=df[['barrels08']]
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)


# train the model
pipeline = Pipeline(steps=[('lr', LinearRegression())])
pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


In [ ]:
# we can see that barrels are strictly 0 for some class of vehicles meaning they might just be EVs
# so let's add that as a feature

# Build a linear regression model
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X=df[df['barrels08']>0][['barrels08']]
y=df[df['barrels08']>0].city08

X_train, X_test, y_train, y_test = train_test_split(X, np.log(y), random_state=42)


# train the model
pipeline = Pipeline(steps=[('lr', LinearRegression())])
pipeline.fit(X_train, y_train)


# evaluate the model

print(f"R2 score is {pipeline.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, pipeline.predict(X_test))}")


In [ ]:
X_train.head